# 🚗 Driving License App – Data Analysis

## 🎯 The aim : 
This notebook presents a structured data pipeline for analyzing user behavior in a driving theory application, from raw data extraction to EDA.

## ⚙️ Project Setup

This section initializes the project environment and imports the required libraries and modules.

In [1]:
%load_ext autoreload
%autoreload 2 

In [2]:
from pathlib import Path
import matplotlib.pyplot as plt
import sys
import polars as pl
import plotly.express as px
import os
from dotenv import load_dotenv

# Project path setup
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\Saloua DOUMI\OneDrive\Documents\internship


## 📥 Data Loading

In this section, raw data is loaded directly from MongoDB and exported to Parquet format written incrementally in batches to improve memory efficiency.

NB : The data is exported to Parquet format because it is a columnar storage format optimized for analytical workloads. It provides better performance and memory efficiency compared to CSV, especially for large datasets.

#### 📦  1. Import Data Loading Functions

In [3]:
from src.data_loading import (
    build_mongo_uri,
    check_mongo_connection,
    get_sample_data,
    export_mongo_to_parquet,
)

#### 🔐 2. Define MongoDB Connection Settings

In [4]:
# Load environment variables from .env file
load_dotenv()

USERNAME = os.getenv("MONGO_USERNAME")
PASSWORD = os.getenv("MONGO_PASSWORD")

# Optional: check if variables are loaded
if not USERNAME or not PASSWORD:
    raise ValueError("Missing MONGO_USERNAME or MONGO_PASSWORD in .env file")

In [5]:

CLUSTER_HOST = "ccd-tech.avpld.mongodb.net"

MONGO_URI = build_mongo_uri(
    username=USERNAME,
    password=PASSWORD,
    cluster_host=CLUSTER_HOST,
)

DB_NAME = "theory"
COLLECTION_NAME = "userAnswers"
OUTPUT_PATH = "data/processed/user_answers_raw.parquet"

#### ✅ 3. Check MongoDB Connection

In [6]:
is_connected = check_mongo_connection(MONGO_URI)
print("MongoDB connection successful:", is_connected)

MongoDB connection successful: True


#### 🔎 4. Load a Small Data Sample

In [7]:
sample_data = get_sample_data(
    uri=MONGO_URI,
    db_name=DB_NAME,
    collection_name=COLLECTION_NAME,
    limit=2,
)

sample_data

[{'_id': ObjectId('63bead4eb3f2c0504beec785'),
  'questionId': '5c61bf1ee12e20001b602de5',
  'userId': '017c44c91350485f80f2def7fbdfaafb',
  'history': [],
  'lastAnswerAt': 1675336298,
  'source': 'list'},
 {'_id': ObjectId('63bedf27b3f2c0504b52c146'),
  'questionId': '5c61bf1ee12e20001b602da5',
  'userId': '017c44c91350485f80f2def7fbdfaafb',
  'history': [{'valid': False,
    'timestamp': 1715720423,
    'context': {'country': 'de',
     'clientType': 'app',
     'appVersion': '1.46.0',
     'demo': False}},
   {'valid': True,
    'timestamp': 1715720397,
    'context': {'country': 'de',
     'clientType': 'app',
     'appVersion': '1.46.0',
     'demo': False}},
   {'valid': True,
    'timestamp': 1689490729,
    'context': {'country': 'de',
     'clientType': 'app',
     'appVersion': '1.39.2',
     'demo': False}},
   {'valid': True,
    'timestamp': 1689490690,
    'context': {'country': 'de',
     'clientType': 'app',
     'appVersion': '1.39.2',
     'demo': False}},
   {'valid

#### 💾 5. Export Raw Data to Parquet

In [8]:
RAW_PARQUET_PATH = PROJECT_ROOT / "data" / "processed" / "user_answers_raw.parquet"
export_mongo_to_parquet(
    uri=MONGO_URI,
    db_name=DB_NAME,
    collection_name=COLLECTION_NAME,
    output_path=str(RAW_PARQUET_PATH),
    batch_size=20_000,
)

Written rows: 20,000
Written rows: 40,000
Written rows: 60,000
Written rows: 80,000
Written rows: 100,000
Written rows: 120,000
Written rows: 140,000
Written rows: 160,000
Written rows: 180,000
Written rows: 200,000
Written rows: 220,000
Written rows: 224,708
Export completed: c:\Users\Saloua DOUMI\OneDrive\Documents\internship\data\processed\user_answers_raw.parquet
Total rows exported: 224,708


## 🔍 Data Understanding

This section explores the structure of the dataset and provides an initial understanding of the key variables, with a focus on the `history` field.

#### 1. Load the raw parquet file

In [9]:
from src.preprocessing import (
    load_raw_parquet,
    parse_history_column,
    add_history_parsing_flags,
    reorder_history_chronologically,

)

RAW_PARQUET_PATH = PROJECT_ROOT/ "data/processed/user_answers_raw.parquet"

lf_raw = load_raw_parquet(RAW_PARQUET_PATH)
lf_raw

#### 2. Preview raw columns

In [10]:
raw_preview = (
    lf_raw
    .select(
        [
            "_id",
            "userId",
            "questionId",
            "history",
            "valid",
            "lastAnswerAt",
            "source",
            "worldId",
        ]
    )
    .head(5)
    .collect()
)

raw_preview

_id,userId,questionId,history,valid,lastAnswerAt,source,worldId
str,str,str,str,bool,str,str,str
"""63bead4eb3f2c0504beec785""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602de5""","""[]""",null,"""1675336298""","""list""",null
"""63bedf27b3f2c0504b52c146""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602da5""","""[{""valid"": false, ""timestamp"":…",false,"""1715720423""","""list""",null
"""63cd0ee96786a82d7d7f2c6e""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602dfd""","""[{""valid"": true, ""timestamp"": …",true,"""1695579765""","""list""",null
"""63d24cef6786a82d7d204cef""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e1c""","""[{""valid"": true, ""timestamp"": …",true,"""1695396341""","""list""",null
"""63d277fc6786a82d7d70a51b""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e27""","""[{""valid"": true, ""timestamp"": …",true,"""1695578238""","""list""",null


#### 3. Understand the raw history field

In [11]:
history_preview = (
    lf_raw
    .select(["history"])
    .head(5)
    .collect()
)

history_preview

history
str
"""[]"""
"""[{""valid"": false, ""timestamp"":…"
"""[{""valid"": true, ""timestamp"": …"
"""[{""valid"": true, ""timestamp"": …"
"""[{""valid"": true, ""timestamp"": …"


## 🔧 Data Transformation

In this section, the raw `history` field is parsed and transformed into a structured format to make it usable for analysis.
- Parsing means converting raw data (such as JSON strings) into a structured format that can be easily analyzed. Using Polars* library
- Polars is a high-performance DataFrame library designed for efficient data processing. It provides faster execution and better scalability than Pandas, especially for large datasets.

#### 1. Parse the history column

In [12]:
lf_parsed = parse_history_column(lf_raw)
lf_parsed

#### 2. Add parsing quality flags

In [13]:
lf_parsed = add_history_parsing_flags(lf_parsed)
lf_parsed

#### 3. Preview the parsed history data

In [14]:
parsed_preview = (
    lf_parsed
    .select(
        [
            "_id",
            "userId",
            "questionId",
            "history",
            "history_parsed",
            "history_parsed_ok",
            "history_length",
        ]
    )
    .head(10)
    .collect()
)

parsed_preview

_id,userId,questionId,history,history_parsed,history_parsed_ok,history_length
str,str,str,str,list[struct[3]],bool,u32
"""63bead4eb3f2c0504beec785""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602de5""","""[]""",[],true,0
"""63bedf27b3f2c0504b52c146""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602da5""","""[{""valid"": false, ""timestamp"":…","[{false,""1715720423"",{""de"",""app"",""1.46.0"",false}}, {true,""1715720397"",{""de"",""app"",""1.46.0"",false}}, … {false,""1689462519"",{""de"",""app"",""1.39.2"",false}}]",true,11
"""63cd0ee96786a82d7d7f2c6e""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602dfd""","""[{""valid"": true, ""timestamp"": …","[{true,""1695579765"",{""de"",""app"",""1.40.0"",false}}, {true,""1695579696"",{""de"",""app"",""1.40.0"",false}}, … {false,""1691291985"",{""de"",""app"",""1.39.2"",false}}]",true,12
"""63d24cef6786a82d7d204cef""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e1c""","""[{""valid"": true, ""timestamp"": …","[{true,""1695396341"",{""de"",""app"",""1.40.0"",false}}, {true,""1695396301"",{""de"",""app"",""1.40.0"",false}}, … {false,""1695396195"",{""de"",""app"",""1.40.0"",false}}]",true,4
"""63d277fc6786a82d7d70a51b""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e27""","""[{""valid"": true, ""timestamp"": …","[{true,""1695578238"",{""de"",""app"",""1.40.0"",false}}, {true,""1695578183"",{""de"",""app"",""1.40.0"",false}}]",true,2
"""63d3a7bd6ae9724d09dd3be5""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e30""","""[{""valid"": true, ""timestamp"": …","[{true,""1695579398"",{""de"",""app"",""1.40.0"",false}}, {false,""1695579365"",{""de"",""app"",""1.40.0"",false}}, … {false,""1695579211"",{""de"",""app"",""1.40.0"",false}}]",true,6
"""63d3ab156ae9724d09e2a3b5""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e3c""","""[{""valid"": true, ""timestamp"": …","[{true,""1695581091"",{""de"",""app"",""1.40.0"",false}}, {false,""1695581081"",{""de"",""app"",""1.40.0"",false}}, {false,""1695581057"",{""de"",""app"",""1.40.0"",false}}]",true,3
"""63d3ab4b6ae9724d09e309f5""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e42""","""[{""valid"": true, ""timestamp"": …","[{true,""1696153683"",{""de"",""app"",""1.40.0"",false}}, {false,""1696153669"",{""de"",""app"",""1.40.0"",false}}, … {true,""1695930174"",{""de"",""app"",""1.40.0"",false}}]",true,4
"""63d3af896ae9724d09ea20a1""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b603042""","""[{""valid"": true, ""timestamp"": …","[{true,""1696368206"",{""de"",""app"",""1.40.0"",false}}, {false,""1696368187"",{""de"",""app"",""1.40.0"",false}}, … {false,""1696365495"",{""de"",""app"",""1.40.0"",false}}]",true,5


NB : The analysis focuses on the `history` field because it contains all user attempts, making it the most important component for understanding user behavior and question difficulty.

#### 4.  Reorder history chronologically

The raw `history` field is stored in reverse chronological order, meaning the most recent attempt appears first and the earliest attempt appears last. To ensure a consistent analytical logic, the parsed history was reordered chronologically before feature engineering.



This reordering step ensures that first-attempt and last-attempt features correctly reflect the actual temporal sequence of user attempts.

In [15]:
lf_parsed = reorder_history_chronologically(lf_parsed)
lf_parsed

## 🧪 Data Quality Checks

This section evaluates the quality of the transformed data and ensures that the parsing process was successful and consistent.

#### 1. Check parsing results

In [16]:
parsing_summary = (
    lf_parsed
    .select(
        [
            pl.len().alias("row_count"),
            pl.col("history_parsed_ok").sum().alias("parsed_rows"),
            pl.col("history_parsed_ok").mean().alias("parsed_rate"),
            pl.col("history_length").mean().alias("avg_history_length"),
            pl.col("history_length").max().alias("max_history_length"),
        ]
    )
    .collect()
)

parsing_summary

row_count,parsed_rows,parsed_rate,avg_history_length,max_history_length
u32,u32,f64,f64,u32
224708,224708,1.0,2.988808,121


- row_count = 224707
→ tu as 224 707 lignes dans ce fichier.
- parsed_rows = 224707
→ toutes les lignes ont été parsées correctement.
- parsed_rate = 1.0
→ taux de succès du parsing = 100%.
- avg_history_length = 2.99
→ en moyenne, chaque ligne contient environ 3 tentatives.
- max_history_length = 121
→ la ligne la plus extrême contient 121 tentatives.

The `history` column was successfully parsed for all rows in the extracted dataset. On average, each user-question pair contains approximately 3 attempts, with a maximum of 121 attempts observed in a single record.

In [17]:
empty_history_stats = (
    lf_parsed
    .select(
        [
            pl.col("history_parsed").list.len().alias("history_length"),
        ]
    )
    .select(
        [
            pl.len().alias("total_rows"),
            (pl.col("history_length") == 0).sum().alias("empty_history_rows"),
            (pl.col("history_length") == 0).mean().alias("empty_history_rate"),
        ]
    )
    .collect()
)

empty_history_stats

total_rows,empty_history_rows,empty_history_rate
u32,u32,f64
224708,34717,0.154498


Some records contain an empty `history` field, meaning no attempts were recorded. These cases were identified during the data quality checks and will be evaluated before deciding whether to include or exclude them from the analysis.

## 🧠 Feature Engineering

#### 1. Create basic features

In [18]:
from src.feature_engineering import (
    add_basic_features,
    add_behavior_features,
    add_time_features,
    add_performance_features,
    build_analytical_dataset,
)

lf_features = add_basic_features(lf_parsed)
lf_features

#### 2. Add behavioral features

In [19]:
lf_features = add_basic_features(lf_parsed)
lf_features = add_behavior_features(lf_features)
lf_features

Additional behavioral features were created to capture how user outcomes evolved across attempts.

These features help identify whether a user improved, declined, or kept the same outcome between the first and the last recorded attempt.

#### 3. Preview engineered features

In [20]:
features_preview = (
    lf_features
    .select(
        [
            "userId",
            "questionId",
            "num_attempts",
            "first_valid",
            "last_valid",
            "is_empty_history",
            "has_multiple_attempts",
            "changed_outcome",
            "improved_between_first_and_last",
            "declined_between_first_and_last",
        ]
    )
    .head(10)
    .collect()
)

features_preview

userId,questionId,num_attempts,first_valid,last_valid,is_empty_history,has_multiple_attempts,changed_outcome,improved_between_first_and_last,declined_between_first_and_last
str,str,u32,bool,bool,bool,bool,bool,bool,bool
"""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602de5""",0,null,null,true,false,null,null,null
"""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602da5""",11,false,false,false,true,false,false,false
"""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602dfd""",12,false,true,false,true,true,true,false
"""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e1c""",4,false,true,false,true,true,true,false
"""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e27""",2,true,true,false,true,false,false,false
"""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e30""",6,false,true,false,true,true,true,false
"""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e3c""",3,false,true,false,true,true,true,false
"""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e42""",4,true,true,false,true,false,false,false
"""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b603042""",5,false,true,false,true,true,true,false


#### 4. Add time-based features

In [21]:
lf_features = add_time_features(lf_features)
lf_features

In [22]:
time_preview = (
    lf_features
    .select(
        [
            "num_attempts",
            "first_timestamp",
            "first_timestamp_dt",
            "last_timestamp",
            "last_timestamp_dt",
            "time_to_last_attempt",
        ]
    )
    .head(10)
    .collect()
)

time_preview

num_attempts,first_timestamp,first_timestamp_dt,last_timestamp,last_timestamp_dt,time_to_last_attempt
u32,str,datetime[μs],str,datetime[μs],i64
0,null,null,null,null,null
11,"""1689462519""",2023-07-15 23:08:39,"""1715720423""",2024-05-14 21:00:23,26257904
12,"""1691291985""",2023-08-06 03:19:45,"""1695579765""",2023-09-24 18:22:45,4287780
4,"""1695396195""",2023-09-22 15:23:15,"""1695396341""",2023-09-22 15:25:41,146
2,"""1695578183""",2023-09-24 17:56:23,"""1695578238""",2023-09-24 17:57:18,55
6,"""1695579211""",2023-09-24 18:13:31,"""1695579398""",2023-09-24 18:16:38,187
3,"""1695581057""",2023-09-24 18:44:17,"""1695581091""",2023-09-24 18:44:51,34
4,"""1695930174""",2023-09-28 19:42:54,"""1696153683""",2023-10-01 09:48:03,223509
5,"""1696365495""",2023-10-03 20:38:15,"""1696368206""",2023-10-03 21:23:26,2711


The timestamp fields were stored as Unix timestamps in string format rather than as ISO datetime strings. They were therefore converted to integers first, then transformed into datetime values for time-based feature engineering.

#### 5. Add performance-related features

In [23]:
lf_features = add_performance_features(lf_features)
lf_features

In [24]:
performance_preview = (
    lf_features
    .select(
        [
            "num_attempts",
            "first_valid",
            "last_valid",
            "answered_correctly_first_try",
            "answered_correctly_last_try",
            "needed_multiple_attempts_to_succeed",
            "never_correct",
            "time_to_last_attempt",
            "avg_time_between_attempts",
        ]
    )
    .head(10)
    .collect()
)

performance_preview

num_attempts,first_valid,last_valid,answered_correctly_first_try,answered_correctly_last_try,needed_multiple_attempts_to_succeed,never_correct,time_to_last_attempt,avg_time_between_attempts
u32,bool,bool,bool,bool,bool,bool,i64,f64
0,null,null,null,null,false,null,null,null
11,false,false,false,false,false,true,26257904,2625790.4
12,false,true,false,true,true,false,4287780,389798.181818
4,false,true,false,true,true,false,146,48.666667
2,true,true,true,true,true,false,55,55.0
6,false,true,false,true,true,false,187,37.4
3,false,true,false,true,true,false,34,17.0
4,true,true,true,true,true,false,223509,74503.0
5,false,true,false,true,true,false,2711,677.75


Performance-related features were created to better describe how users interacted with each question across multiple attempts.



These variables help distinguish between immediate success, delayed success, repeated failure, and the average time spent between attempts.

#### 6.  Build the final analytical dataset

In [51]:
lf_analytical = build_analytical_dataset(lf_features)
lf_analytical

#### 7. Preview the final analytical dataset

In [52]:
analytical_preview = lf_analytical.head(10).collect()
analytical_preview

_id,userId,questionId,source,worldId,lastAnswerAt,lastContext_country,lastContext_clientType,lastContext_appVersion,lastContext_demo,num_attempts,first_valid,last_valid,first_timestamp,last_timestamp,first_timestamp_dt,last_timestamp_dt,time_to_last_attempt,is_empty_history,has_multiple_attempts,changed_outcome,improved_between_first_and_last,declined_between_first_and_last,answered_correctly_first_try,answered_correctly_last_try,needed_multiple_attempts_to_succeed,never_correct,avg_time_between_attempts,history_ordered
str,str,str,str,str,str,str,str,str,bool,u32,bool,bool,str,str,datetime[μs],datetime[μs],i64,bool,bool,bool,bool,bool,bool,bool,bool,bool,f64,list[struct[3]]
"""63bead4eb3f2c0504beec785""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602de5""","""list""",null,"""1675336298""",null,null,null,null,0,null,null,null,null,null,null,null,true,false,null,null,null,null,null,false,null,null,[]
"""63bedf27b3f2c0504b52c146""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602da5""","""list""",null,"""1715720423""","""de""","""app""","""1.46.0""",false,11,false,false,"""1689462519""","""1715720423""",2023-07-15 23:08:39,2024-05-14 21:00:23,26257904,false,true,false,false,false,false,false,false,true,2625790.4,"[{false,""1689462519"",{""de"",""app"",""1.39.2"",false}}, {false,""1689490477"",{""de"",""app"",""1.39.2"",false}}, … {false,""1715720423"",{""de"",""app"",""1.46.0"",false}}]"
"""63cd0ee96786a82d7d7f2c6e""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602dfd""","""list""",null,"""1695579765""","""de""","""app""","""1.40.0""",false,12,false,true,"""1691291985""","""1695579765""",2023-08-06 03:19:45,2023-09-24 18:22:45,4287780,false,true,true,true,false,false,true,true,false,389798.181818,"[{false,""1691291985"",{""de"",""app"",""1.39.2"",false}}, {false,""1691292003"",{""de"",""app"",""1.39.2"",false}}, … {true,""1695579765"",{""de"",""app"",""1.40.0"",false}}]"
"""63d24cef6786a82d7d204cef""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e1c""","""list""",null,"""1695396341""","""de""","""app""","""1.40.0""",false,4,false,true,"""1695396195""","""1695396341""",2023-09-22 15:23:15,2023-09-22 15:25:41,146,false,true,true,true,false,false,true,true,false,48.666667,"[{false,""1695396195"",{""de"",""app"",""1.40.0"",false}}, {false,""1695396246"",{""de"",""app"",""1.40.0"",false}}, … {true,""1695396341"",{""de"",""app"",""1.40.0"",false}}]"
"""63d277fc6786a82d7d70a51b""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e27""","""list""",null,"""1695578238""","""de""","""app""","""1.40.0""",false,2,true,true,"""1695578183""","""1695578238""",2023-09-24 17:56:23,2023-09-24 17:57:18,55,false,true,false,false,false,true,true,true,false,55.0,"[{true,""1695578183"",{""de"",""app"",""1.40.0"",false}}, {true,""1695578238"",{""de"",""app"",""1.40.0"",false}}]"
"""63d3a7bd6ae9724d09dd3be5""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e30""","""list""",null,"""1695579398""","""de""","""app""","""1.40.0""",false,6,false,true,"""1695579211""","""1695579398""",2023-09-24 18:13:31,2023-09-24 18:16:38,187,false,true,true,true,false,false,true,true,false,37.4,"[{false,""1695579211"",{""de"",""app"",""1.40.0"",false}}, {false,""1695579235"",{""de"",""app"",""1.40.0"",false}}, … {true,""1695579398"",{""de"",""app"",""1.40.0"",false}}]"
"""63d3ab156ae9724d09e2a3b5""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e3c""","""list""",null,"""1695581091""","""de""","""app""","""1.40.0""",false,3,false,true,"""1695581057""","""1695581091""",2023-09-24 18:44:17,2023-09-24 18:44:51,34,false,true,true,true,false,false,true,true,false,17.0,"[{false,""1695581057"",{""de"",""app"",""1.40.0"",false}}, {false,""1695581081"",{""de"",""app"",""1.40.0"",false}}, {true,""1695581091"",{""de"",""app"",""1.40.0"",false}}]"
"""63d3ab4b6ae9724d09e309f5""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e42""","""list""",null,"""1696153683""","""de""","""app""","

- ``first_valid = null
last_valid = null
→ changed_outcome = null`` Outcome-based features are only defined when at least one attempt is available.

At this stage, a final analytical dataset was created by selecting the most relevant raw and engineered variables for downstream analysis.

This dataset provides a clean and reusable table at the user-question level, which will be used for aggregation and exploratory analysis.

## 📈 Exploratory Data Analysis

#### 1. Global understanding

##### 1.1  Compute global summary metrics

In [53]:
global_summary = (
    lf_analytical
    .select(
        [
            pl.len().alias("row_count"),
            pl.col("userId").n_unique().alias("unique_users"),
            pl.col("questionId").n_unique().alias("unique_questions"),
            pl.col("num_attempts").mean().alias("avg_num_attempts"),
            pl.col("num_attempts").median().alias("median_num_attempts"),
            pl.col("last_valid").mean().alias("final_success_rate"),
            pl.col("first_valid").mean().alias("first_try_success_rate"),
            pl.col("has_multiple_attempts").mean().alias("multiple_attempt_rate"),
            pl.col("never_correct").mean().alias("never_correct_rate"),
            pl.col("time_to_last_attempt").mean().alias("avg_time_to_last_attempt"),
            pl.col("time_to_last_attempt").median().alias("median_time_to_last_attempt"),
        ]
    )
    .collect()
)

global_summary

row_count,unique_users,unique_questions,avg_num_attempts,median_num_attempts,final_success_rate,first_try_success_rate,multiple_attempt_rate,never_correct_rate,avg_time_to_last_attempt,median_time_to_last_attempt
u32,u32,u32,f64,f64,f64,f64,f64,f64,f64,f64
224708,342,4680,2.988808,2.0,0.950087,0.713844,0.6007,0.049913,2.1874e6,542.0


##### 1.2 Compute question-level metrics

In [54]:
question_summary = (
    lf_analytical
    .group_by("questionId")
    .agg(
        [
            pl.len().alias("row_count"),
            pl.col("num_attempts").mean().alias("avg_num_attempts"),
            pl.col("num_attempts").median().alias("median_num_attempts"),
            pl.col("first_valid").mean().alias("first_try_success_rate"),
            pl.col("last_valid").mean().alias("final_success_rate"),
            pl.col("has_multiple_attempts").mean().alias("multiple_attempt_rate"),
            pl.col("never_correct").mean().alias("never_correct_rate"),
            pl.col("time_to_last_attempt").median().alias("median_time_to_last_attempt"),
        ]
    )
    .sort("row_count", descending=True)
    .collect()
)

question_summary.head(10)

questionId,row_count,avg_num_attempts,median_num_attempts,first_try_success_rate,final_success_rate,multiple_attempt_rate,never_correct_rate,median_time_to_last_attempt
str,u32,f64,f64,f64,f64,f64,f64,f64
"""5c61bf1ee12e20001b602d9f""",404,3.064356,1.0,0.819718,0.921127,0.480198,0.078873,133.0
"""5c61bf1ee12e20001b602da0""",380,3.357895,1.0,0.851515,0.918182,0.486842,0.081818,209.0
"""5c61bf1ee12e20001b602d9d""",378,3.587302,1.0,0.645455,0.887879,0.481481,0.112121,140.0
"""5c61bf1ee12e20001b602d9e""",378,3.013228,1.0,0.911854,0.954407,0.481481,0.045593,157.0
"""5c61bf1ee12e20001b602da2""",372,2.951613,1.0,0.907692,0.950769,0.454301,0.049231,55.5
"""5c61bf1ee12e20001b602e3f""",357,2.588235,1.0,0.821656,0.955414,0.467787,0.044586,61.0
"""5c61bf1ee12e20001b602e38""",340,2.729412,1.0,0.908475,0.966102,0.45,0.033898,50.5
"""5c61bf1ee12e20001b602e40""",333,2.633634,1.0,0.878049,0.95122,0.426426,0.04878,0.0
"""5c61bf1ee12e20001b602d9a""",328,3.408537,2.0,0.791367,0.946043,0.536585,0.053957,681.0


Initial aggregation results show that most users eventually succeed, but a significant proportion requires multiple attempts, indicating a non-trivial level of difficulty across questions.

The large difference between mean and median time suggests the presence of extreme values, likely caused by users returning to questions after long periods.

**This section explores the analytical dataset to identify patterns related to question difficulty, user behavior, and time dynamics.**

In [55]:
df = lf_analytical.collect()

#### 2. User behavior

**How many attempts do users need to answer a question?**

In [56]:
attempts_dist = (
    df.filter(pl.col("num_attempts") <= 15)
    .group_by("num_attempts")
    .count()
    .sort("num_attempts")
)

fig = px.bar(
    attempts_dist.to_pandas(),
    x="num_attempts",
    y="count",
    title="Distribution of number of attempts per question"
)

fig.show()

C:\Users\Saloua DOUMI\AppData\Local\Temp\ipykernel_20504\4040505077.py:4: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()


Each observation represents a user-question interaction. This distribution shows how many attempts users typically make when answering a question.

The distribution is highly right-skewed. Most interactions require only one or two attempts, while a smaller number involve many repeated attempts.

This suggests that most questions are relatively easy, but some interactions reflect either difficult questions or persistent user behavior.

#### 3. Question difficulty

**which questions are the most difficult?**

In [57]:
df_analysis = df.filter(pl.col("num_attempts") > 0)

print(f"Number of rows after filtering: {len(df_analysis)}")
df_analysis.head(3)

Number of rows after filtering: 189991


_id,userId,questionId,source,worldId,lastAnswerAt,lastContext_country,lastContext_clientType,lastContext_appVersion,lastContext_demo,num_attempts,first_valid,last_valid,first_timestamp,last_timestamp,first_timestamp_dt,last_timestamp_dt,time_to_last_attempt,is_empty_history,has_multiple_attempts,changed_outcome,improved_between_first_and_last,declined_between_first_and_last,answered_correctly_first_try,answered_correctly_last_try,needed_multiple_attempts_to_succeed,never_correct,avg_time_between_attempts,history_ordered
str,str,str,str,str,str,str,str,str,bool,u32,bool,bool,str,str,datetime[μs],datetime[μs],i64,bool,bool,bool,bool,bool,bool,bool,bool,bool,f64,list[struct[3]]
"""63bedf27b3f2c0504b52c146""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602da5""","""list""",null,"""1715720423""","""de""","""app""","""1.46.0""",false,11,false,false,"""1689462519""","""1715720423""",2023-07-15 23:08:39,2024-05-14 21:00:23,26257904,false,true,false,false,false,false,false,false,true,2625790.4,"[{false,""1689462519"",{""de"",""app"",""1.39.2"",false}}, {false,""1689490477"",{""de"",""app"",""1.39.2"",false}}, … {false,""1715720423"",{""de"",""app"",""1.46.0"",false}}]"
"""63cd0ee96786a82d7d7f2c6e""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602dfd""","""list""",null,"""1695579765""","""de""","""app""","""1.40.0""",false,12,false,true,"""1691291985""","""1695579765""",2023-08-06 03:19:45,2023-09-24 18:22:45,4287780,false,true,true,true,false,false,true,true,false,389798.181818,"[{false,""1691291985"",{""de"",""app"",""1.39.2"",false}}, {false,""1691292003"",{""de"",""app"",""1.39.2"",false}}, … {true,""1695579765"",{""de"",""app"",""1.40.0"",false}}]"
"""63d24cef6786a82d7d204cef""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e1c""","""list""",null,"""1695396341""","""de""","""app""","""1.40.0""",false,4,false,true,"""1695396195""","""1695396341""",2023-09-22 15:23:15,2023-09-22 15:25:41,146,false,true,true,true,false,false,true,true,false,48.666667,"[{false,""1695396195"",{""de"",""app"",""1.40.0"",false}}, {false,""1695396246"",{""de"",""app"",""1.40.0"",false}}, … {true,""1695396341"",{""de"",""app"",""1.40.0"",false}}]"


In [58]:


question_stats = (
    df_analysis
    .group_by("questionId")
    .agg([
        pl.col("first_valid").mean().alias("first_try_success_rate"),
        pl.col("num_attempts").mean().alias("avg_attempts")
    ])
)

In [59]:
fig = px.scatter(
    question_stats.filter(pl.col("avg_attempts") <= 10),
    x="avg_attempts",
    y="first_try_success_rate",
    opacity=0.6,
    title="Question difficulty: attempts vs success rate"
)

fig.show()

- ``Top left``: easy questions (high success rate, few attempts)

- ``Bottom left``: low engagement or quick drop-off (low success, few attempts)

- ``Top right``: atypical behavior (high success despite many attempts)

- ``Bottom right``: difficult questions (low success rate, many attempts)

The plot shows a clear relationship between the number of attempts and the success rate.

Questions that require more attempts tend to have lower first-try success rates, indicating higher difficulty.

#### 4. User learning behavior

**Do users improve after multiple attempts?**

In [60]:
behavior_counts = (
    df_analysis
    .select([
        pl.col("improved_between_first_and_last").sum().alias("Improved"),
        pl.col("declined_between_first_and_last").sum().alias("Declined"),
        (pl.col("changed_outcome") == False).sum().alias("Unchanged")
    ])
)

behavior_counts

Improved,Declined,Unchanged
u32,u32,u32
46389,1505,142097


In [61]:
behavior_df = behavior_counts.to_pandas().T.reset_index()
behavior_df.columns = ["Category", "Count"]

behavior_df["Percentage"] = (
    behavior_df["Count"] / behavior_df["Count"].sum() * 100
).round(1)

fig = px.bar(
    behavior_df,
    x="Category",
    y="Count",
    text="Percentage",
    title="User learning behavior"
)

fig.show()

Most users either improve or remain consistent across attempts, while only a small proportion show declining performance.

Unchanged outcomes include both consistently correct and consistently incorrect answers, and therefore represent stable behavior rather than improvement or decline.

#### 5. Time analysis

**How much time do users take between attempts?**

In [62]:
df_time = df_analysis.filter(pl.col("time_to_last_attempt").is_not_null())

In [63]:
df_time

_id,userId,questionId,source,worldId,lastAnswerAt,lastContext_country,lastContext_clientType,lastContext_appVersion,lastContext_demo,num_attempts,first_valid,last_valid,first_timestamp,last_timestamp,first_timestamp_dt,last_timestamp_dt,time_to_last_attempt,is_empty_history,has_multiple_attempts,changed_outcome,improved_between_first_and_last,declined_between_first_and_last,answered_correctly_first_try,answered_correctly_last_try,needed_multiple_attempts_to_succeed,never_correct,avg_time_between_attempts,history_ordered
str,str,str,str,str,str,str,str,str,bool,u32,bool,bool,str,str,datetime[μs],datetime[μs],i64,bool,bool,bool,bool,bool,bool,bool,bool,bool,f64,list[struct[3]]
"""63bedf27b3f2c0504b52c146""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602da5""","""list""",null,"""1715720423""","""de""","""app""","""1.46.0""",false,11,false,false,"""1689462519""","""1715720423""",2023-07-15 23:08:39,2024-05-14 21:00:23,26257904,false,true,false,false,false,false,false,false,true,2625790.4,"[{false,""1689462519"",{""de"",""app"",""1.39.2"",false}}, {false,""1689490477"",{""de"",""app"",""1.39.2"",false}}, … {false,""1715720423"",{""de"",""app"",""1.46.0"",false}}]"
"""63cd0ee96786a82d7d7f2c6e""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602dfd""","""list""",null,"""1695579765""","""de""","""app""","""1.40.0""",false,12,false,true,"""1691291985""","""1695579765""",2023-08-06 03:19:45,2023-09-24 18:22:45,4287780,false,true,true,true,false,false,true,true,false,389798.181818,"[{false,""1691291985"",{""de"",""app"",""1.39.2"",false}}, {false,""1691292003"",{""de"",""app"",""1.39.2"",false}}, … {true,""1695579765"",{""de"",""app"",""1.40.0"",false}}]"
"""63d24cef6786a82d7d204cef""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e1c""","""list""",null,"""1695396341""","""de""","""app""","""1.40.0""",false,4,false,true,"""1695396195""","""1695396341""",2023-09-22 15:23:15,2023-09-22 15:25:41,146,false,true,true,true,false,false,true,true,false,48.666667,"[{false,""1695396195"",{""de"",""app"",""1.40.0"",false}}, {false,""1695396246"",{""de"",""app"",""1.40.0"",false}}, … {true,""1695396341"",{""de"",""app"",""1.40.0"",false}}]"
"""63d277fc6786a82d7d70a51b""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e27""","""list""",null,"""1695578238""","""de""","""app""","""1.40.0""",false,2,true,true,"""1695578183""","""1695578238""",2023-09-24 17:56:23,2023-09-24 17:57:18,55,false,true,false,false,false,true,true,true,false,55.0,"[{true,""1695578183"",{""de"",""app"",""1.40.0"",false}}, {true,""1695578238"",{""de"",""app"",""1.40.0"",false}}]"
"""63d3a7bd6ae9724d09dd3be5""","""017c44c91350485f80f2def7fbdfaa…","""5c61bf1ee12e20001b602e30""","""list""",null,"""1695579398""","""de""","""app""","""1.40.0""",false,6,false,true,"""1695579211""","""1695579398""",2023-09-24 18:13:31,2023-09-24 18:16:38,187,false,true,true,true,false,false,true,true,false,37.4,"[{false,""1695579211"",{""de"",""app"",""1.40.0"",false}}, {false,""1695579235"",{""de"",""app"",""1.40.0"",false}}, … {true,""1695579398"",{""de"",""app"",""1.40.0"",false}}]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""69ac92482847c9a97f31858d""","""d5f44343441b436caf495690a5f538…","""5c61bf1ee12e20001b603056""","""list""",null,"""1772917319""",null,null,null,null,1,true,true,"""1772917319""","""1772917319""",2026-03-07 21:01:59,2026-03-07 21:01:59,0,false,false,false,false,false,true,true,false,false,null,"[{true,""1772917319"",null}]"
"""69ac92482847c9a97f31858e""","""d5f44343441b436caf495690a5f538…","""5c61bf1ee12e20001b602f27""","""list""",null,"""1772917319""",null,null,null,null,1,true,true,"""1772917319""","""1772917319""",2026-03-07 21:01:59,2026-03-07 21:01:59,0,false,false,false,false,false,true,true,false,false,null,"[{true,""1772917319"",null}]"
"""69b2c0b278778b07c16d3693""","""699f0fc0cbbb44899d17e4752a7562…","""5c61bf1ee12e20001b603044""","""world""","""67e2e3b0c97c0b775183c306""","""1773322416""",null,null,nul

In [64]:
fig = px.histogram(
    df_time.to_pandas(),
    x="time_to_last_attempt",
    nbins=50,
    title="Distribution of time between first and last attempt"
)

fig.show()

In [65]:
df_time = df_analysis.filter(pl.col("time_to_last_attempt") < 3600)
fig = px.histogram(
    df_time.to_pandas(),
    x="time_to_last_attempt",
    nbins=50,
    title="Time between first and last attempt (under 1 hour)"
)

fig.show()

#### 6. Question Difficulty vs Response Time

**Is there a relationship between how difficult a question is and how much time users spend on it?**

This visualization helps us understand whether difficult questions correlate with longer response times.
We'll create two categorical variables:
- `difficulty_bucket`: Easy, Medium, Difficult (based on average attempts)
- `time_bucket`: Fast, Medium, Slow (based on time between first and last attempt)

##### 6.1 Difficulty buckets

- Questions are grouped into three categories based on the number of attempts users needed:
  - `easy`: Questions with the lowest number of attempts (bottom 33rd percentile)
  - `medium`: Questions with average attempts (33rd to 66th percentile)
  - `difficult`: Questions with the highest number of attempts (top 66th percentile)
- This ensures each category contains roughly one-third of all questions.


In [66]:
df_analysis = df_analysis.with_columns([
    pl.col("num_attempts")
    .qcut([0.33, 0.66], labels=["easy", "medium", "difficult"])
    .alias("difficulty_bucket")
])

# Check the new column
print("Difficulty buckets created:")
df_analysis.select(["num_attempts", "difficulty_bucket"]).head(10)

Difficulty buckets created:


num_attempts,difficulty_bucket
u32,cat
11,"""difficult"""
12,"""difficult"""
4,"""difficult"""
2,"""easy"""
6,"""difficult"""
3,"""medium"""
4,"""difficult"""
5,"""difficult"""
10,"""difficult"""


In [67]:
# See how many questions in each difficulty bucket
difficulty_counts = df_analysis.group_by("difficulty_bucket").agg(pl.len().alias("count"))
print("Distribution of difficulty buckets:")
difficulty_counts

Distribution of difficulty buckets:


difficulty_bucket,count
cat,u32
"""difficult""",58141
"""easy""",105005
"""medium""",26845


##### 6.2 Response time buckets
- Response time is measured as `time_to_last_attempt` (seconds between first and last attempt)
- Questions are grouped into three categories:
  - `fast`: Shortest response times (bottom 33rd percentile)
  - `medium`: Average response times (33rd to 66th percentile)
  - `slow`: Longest response times (top 66th percentile)

In [68]:

df_time = df_analysis.filter(pl.col("time_to_last_attempt").is_not_null())

print(f"Rows with time data: {len(df_time)}")

# Create time buckets
df_time = df_time.with_columns([
    pl.col("time_to_last_attempt")
    .qcut([0.33, 0.66], labels=["fast", "medium", "slow"])
    .alias("time_bucket")
])

# Check the result
df_time.select(["time_to_last_attempt", "time_bucket", "difficulty_bucket"]).head(10)

Rows with time data: 189603


time_to_last_attempt,time_bucket,difficulty_bucket
i64,cat,cat
26257904,"""slow""","""difficult"""
4287780,"""slow""","""difficult"""
146,"""medium""","""difficult"""
55,"""fast""","""easy"""
187,"""medium""","""difficult"""
34,"""fast""","""medium"""
223509,"""slow""","""difficult"""
2711,"""medium""","""difficult"""
209601,"""slow""","""difficult"""


##### 6.3 Build the Heatmap

In [69]:
heatmap_data = (
    df_time
    .group_by(["difficulty_bucket", "time_bucket"])
    .agg(pl.len().alias("count"))
    .sort("difficulty_bucket", "time_bucket")
)

print("Heatmap data ready:")
heatmap_data

Heatmap data ready:


difficulty_bucket,time_bucket,count
cat,cat,u32
"""difficult""","""fast""",333
"""difficult""","""medium""",14540
"""difficult""","""slow""",43268
"""easy""","""fast""",61617
"""easy""","""medium""",33161
"""easy""","""slow""",9839
"""medium""","""fast""",712
"""medium""","""medium""",14780
"""medium""","""slow""",11353


In [70]:
heatmap_pandas = heatmap_data.to_pandas()

# Define the correct order
difficulty_order = ["easy", "medium", "difficult"]
time_order = ["fast", "medium", "slow"]

fig = px.density_heatmap(
    heatmap_pandas,
    x="difficulty_bucket",
    y="time_bucket",
    z="count",
    title="Question Difficulty vs Response Time",
    labels={
        "difficulty_bucket": "Question Difficulty",
        "time_bucket": "Response Time",
        "count": "Number of Questions"
    },
    color_continuous_scale="Viridis",
    category_orders={
        "difficulty_bucket": difficulty_order,
        "time_bucket": time_order
    }
)

fig.show()

- Each cell represents the number of questions that fall into a specific combination of difficulty and response time
- Color intensity follows the scale on the right:
  - **Yellow** = Highest number of questions
  - **Green/Purple** = Medium number of questions
  - **Dark Blue** = Lowest number of questions

**How to Read the Heatmap ?**

| Cell | Color | Interpretation |
|------|-------|----------------|
| **fast + easy** | 🟡 Yellow (maximum) | ✅ **Expected**: The largest group of questions are easy and answered quickly. This validates our difficulty classification. |
| **slow + difficult** | 🟢 Light green (very high) | ✅ **Expected**: Many difficult questions take time to answer. Users are spending time thinking on challenging questions. |
| **medium + easy** | 🟢 Green (high) | ✅ **Expected**: A significant number of easy questions are answered in average time. |
| **fast + difficult** | 🟣 Purple (medium) | ✅ **Expected**: Some difficult questions are answered quickly. This is normal as not all "difficult" questions are equally hard. |
| **fast + medium** | 🟣 Purple (medium) | ✅ **Expected**: A moderate number of medium-difficulty questions are answered quickly. |
| **slow + easy** | 🔵 Dark blue (lowest) | ✅ **Expected**: Very few easy questions take a long time. |

*Conclusion*
   - Easy questions → answered quickly (yellow and green in the fast/medium rows)
   - Difficult questions → often take time (light green in the slow row)
   - Very few easy questions take a long time (dark blue at `slow + easy`)


### 8. Activity and performance 

##### 8.1 Activity by Day of Week

In [96]:

activity_weekday = (
    df_time_features
    .with_columns([
        pl.col("first_timestamp_dt").dt.strftime("%A").alias("weekday_name")
    ])
    .group_by("weekday_name")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("percentage")
    )
    .sort("weekday_name")
)

# Create a label with both count and percentage
activity_weekday_df = activity_weekday.to_pandas()
activity_weekday_df["label"] = activity_weekday_df["percentage"].astype(str) + "%"

fig = px.bar(
    activity_weekday_df,
    x="weekday_name",
    y="count",
    title="User Activity by Day of Week",
    labels={"count": "Number of Interactions", "weekday_name": "Day"},
    text="label"
)
fig.update_traces(textposition='outside')
fig.show()

Activity is evenly distributed across weekdays, with Thursday (15.5%) and Sunday (15.3%) showing the highest engagement. Saturday (11.6%) is the least active day.

##### 8.2 Activity by Season

In [93]:
# Activity by season with count and percentage
activity_season = (
    df_time_features
    .group_by("season")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("percentage")
    )
    .sort("season")
)

activity_season_df = activity_season.to_pandas()
activity_season_df["label"] =  activity_season_df["percentage"].astype(str) + "%"

fig = px.bar(
    activity_season_df,
    x="season",
    y="count",
    title="User Activity by Season",
    labels={"count": "Number of Interactions", "season": "Season"},
    text="label"
)
fig.update_traces(textposition='outside')
fig.show()

Spring (44.9%) accounts for nearly half of all interactions, followed by Winter (31.2%). Summer and Autumn show lower activity levels.

In [100]:
activity_season_df

,season,count,percentage,label
0,Autumn,24496,12.9,12.9%
1,Spring,85112,44.9,44.9%
2,Summer,20896,11.0,11.0%
3,Winter,59099,31.2,31.2%


##### 8.3 Activity by Weekend/Weekday 

In [95]:
# Activity by weekend/weekday with count and percentage
activity_period = (
    df_time_features
    .group_by("is_weekend")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("percentage")
    )
)

activity_period_df = activity_period.to_pandas()
activity_period_df["is_weekend"] = activity_period_df["is_weekend"].map({True: "Weekend", False: "Weekday"})
activity_period_df["label"] =activity_period_df["percentage"].astype(str) + "%"

fig = px.bar(
    activity_period_df,
    x="is_weekend",
    y="count",
    title="User Activity: Weekend vs Weekday",
    labels={"count": "Number of Interactions", "is_weekend": "Period"},
    text="label"
)
fig.update_traces(textposition='outside')
fig.show()

The majority of activity occurs on weekdays (73%), while weekends account for 27% of interactions.

##### 8.4  Activity by Time of Day

In [98]:
activity_time = (
    df_time_features
    .group_by("time_of_day")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("percentage")
    )
    .sort("time_of_day")
)

activity_time_df = activity_time.to_pandas()
activity_time_df["label"] =  activity_time_df["percentage"].astype(str) + "%"

fig = px.bar(
    activity_time_df,
    x="time_of_day",
    y="count",
    title="User Activity by Time of Day",
    labels={"count": "Number of Interactions", "time_of_day": "Time of Day"},
    text="label"
)
fig.update_traces(textposition='outside')
fig.show()

Afternoon (34.1%) and Evening (33.4%) are the peak periods, together representing two-thirds of all activity. Morning accounts for 25.1%, while night activity is minimal (7.4%)

**Key Insight**: Users are most active on weekdays, particularly on Thursday and Sunday, during afternoon and evening hours, with a strong peak in Spring season.

### 9. Users and performance

##### Percentage of users by season

In [ ]:
users_by_season = (
    df_time_features
    .group_by("season")
    .agg(pl.col("userId").n_unique().alias("unique_users"))
    .with_columns(
        (pl.col("unique_users") / pl.col("unique_users").sum() * 100).round(1).alias("user_percentage")
    )
    .sort("season")
)

print("Unique users by season:")
users_by_season

Unique users by season:


season,unique_users,user_percentage
str,u32,f64
"""Autumn""",67,12.5
"""Spring""",184,34.3
"""Summer""",131,24.4
"""Winter""",154,28.7


##### 9.2 Average interactions per user by season

In [ ]:

avg_activity_by_season = (
    df_time_features
    .group_by(["season", "userId"])
    .agg(pl.len().alias("user_interactions"))
    .group_by("season")
    .agg(pl.col("user_interactions").mean().round(1).alias("avg_interactions_per_user"))
    .sort("season")
)

print("Average interactions per user by season:")
avg_activity_by_seasons:vmoSIJDIDDUU8QY7<Y7777S7Yqyçijoioàào_uç_ywjwh

Average interactions per user by season:


season,avg_interactions_per_user
str,f64
"""Autumn""",365.6
"""Spring""",462.6
"""Summer""",159.5
"""Winter""",383.8


**Recommendations**


**Summer**: Low Engagement

Observation: 24% of users are active in summer, but they complete 3x fewer exercises than in spring (159 vs 462 interactions per user).

What we can do:

    Summer challenge: Offer a special badge for completing 50 questions in July/August

    Short sessions: Propose 5-minute daily reviews (easy to fit between activities)

    Light notifications: One reminder per week, not more

**Autumn**: Loyal but Few Users

Observation: Only 12.5% of users are active in autumn, but they are very loyal (365 interactions per user).

What we can do:

    Reward loyalty: Send a "Thank you for being with us" message

    Referral program: "Invite a friend" to grow this loyal group

**Spring**: Peak Activity

Observation: 34% of users are active in spring with 462 interactions per user — the best period.

What we can do:

    Launch new features in spring (new questions, new content)

    Marketing campaigns to attract new users before summer



## Learning Progress Analyis